## LeNet5

Outputs

- `models/lenet5_quantized.tflite` INT8 input/output, per-tensor; MicroFlow-compatible

Keep `epochs` small if you only need artifacts quickly.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import tensorflow as tf

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")

try:
    tf.get_logger().setLevel("ERROR")
except Exception:
    pass

REPO_ROOT = Path.cwd().parent
MODELS_DIR = REPO_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

OUT_TFLITE_QUANT = MODELS_DIR / "lenet5_quantized_tf.tflite"

print("OUT_TFLITE_QUANT:", OUT_TFLITE_QUANT)

I0000 00:00:1776950224.559319   29867 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776950224.824948   29867 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1776950228.839198   29867 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


OUT_TFLITE_QUANT: /home/nathan/Documents/ariel-microflow-ml/models/lenet5_quantized.tflite


In [2]:
def build_lenet5() -> tf.keras.Model:
    inputs = tf.keras.Input(shape=(28, 28, 1), batch_size=1, name="input")
    x = tf.keras.layers.Conv2D(6, (5, 5), activation="relu", padding="valid")(inputs)
    x = tf.keras.layers.AveragePooling2D(pool_size=(2, 2), strides=(2, 2))(x)
    x = tf.keras.layers.Conv2D(16, (5, 5), activation="relu", padding="valid")(x)
    x = tf.keras.layers.AveragePooling2D(pool_size=(2, 2), strides=(2, 2))(x)
    x = tf.keras.layers.Reshape((256,), name="flatten_256")(x)
    x = tf.keras.layers.Dense(120, activation="relu", name="fc1")(x)
    x = tf.keras.layers.Dense(84, activation="relu", name="fc2")(x)
    logits = tf.keras.layers.Dense(10, name="fc3")(x)
    outputs = tf.keras.layers.Softmax(name="softmax")(logits)
    return tf.keras.Model(inputs=inputs, outputs=outputs, name="lenet5")

model = build_lenet5()
model.summary()

W0000 00:00:1776950231.743134   29867 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Model: "lenet5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (1, 28, 28, 1)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (1, 24, 24, 6)         │           156 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d               │ (1, 12, 12, 6)         │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (1, 8, 8, 16)          │         2,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_1             │ (1, 4, 4, 16)          │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_256 (Reshape)           │ (1, 256)               │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (1, 120)               │        30,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc2 (Dense)                     │ (1, 84)                │        10,164 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc3 (Dense)                     │ (1, 10)                │           850 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ softmax (Softmax)               │ (1, 10)                │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 44,426 (173.54 KB)

 Trainable params: 44,426 (173.54 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:

epochs = 4
batch_size = 128

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = (x_train.astype(np.float32) / 255.0)[..., None]
x_test = (x_test.astype(np.float32) / 255.0)[..., None]

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.fit(
    x_train,
    y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1,
    verbose=2,
)

print("Test accuracy:", model.evaluate(x_test, y_test, verbose=0)[1])

Epoch 1/2
422/422 - 3s - 8ms/step - accuracy: 0.8732 - loss: 0.4276 - val_accuracy: 0.9525 - val_loss: 0.1564
Epoch 2/2
422/422 - 2s - 5ms/step - accuracy: 0.9589 - loss: 0.1357 - val_accuracy: 0.9720 - val_loss: 0.0901
Test accuracy: 0.972000002861023


In [4]:
def representative_data_gen():
    for i in range(200):
        yield [x_train[i : i + 1].astype(np.float32)]

int8_converter = tf.lite.TFLiteConverter.from_keras_model(model)
int8_converter.optimizations = [tf.lite.Optimize.DEFAULT]
int8_converter.representative_dataset = representative_data_gen
int8_converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
int8_converter.target_spec.supported_types = [tf.int8]
int8_converter.inference_input_type = tf.int8
int8_converter.inference_output_type = tf.int8

for attr in (
    "_experimental_disable_per_channel",
    "_experimental_disable_per_channel_quantization",
):
    if hasattr(int8_converter, attr):
        setattr(int8_converter, attr, True)

for attr in ("experimental_new_quantizer", "_experimental_new_quantizer"):
    if hasattr(int8_converter, attr):
        setattr(int8_converter, attr, False)

int8_tflite = int8_converter.convert()
OUT_TFLITE_QUANT.write_bytes(int8_tflite)

int8_interp = tf.lite.Interpreter(model_path=str(OUT_TFLITE_QUANT), experimental_delegates=[])
int8_interp.allocate_tensors()
in_info = int8_interp.get_input_details()[0]
out_info = int8_interp.get_output_details()[0]

print("wrote", OUT_TFLITE_QUANT, "bytes=", OUT_TFLITE_QUANT.stat().st_size)
print("int8 input:", in_info["dtype"], in_info["shape"], "quant=", in_info["quantization"])
print("int8 output:", out_info["dtype"], out_info["shape"], "quant=", out_info["quantization"])

per_channel = []
for td in int8_interp.get_tensor_details():
    qp = td.get("quantization_parameters") or {}
    scales = qp.get("scales")
    if isinstance(scales, np.ndarray) and scales.size > 1:
        per_channel.append((td.get("name", "?"), int(scales.size)))

if per_channel:
    print("Warning: per-channel quantization detected (first 20):")
    for name, n in per_channel[:20]:
        print(" -", name, "scales=", n)
else:
    print("Per-channel quantization check: OK (all tensors per-tensor or unquantized)")


Saved artifact at '/tmp/tmp3287fu2m'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 28, 28, 1), dtype=tf.float32, name='input')
Output Type:
  TensorSpec(shape=(1, 10), dtype=tf.float32, name=None)
Captures:
  130735299996624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130735299997584: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130735299997968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130735299998544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130735299999888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130735299997776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130735299996816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130735299999120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130735299998928: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130735299998352: TensorSpec(shape=(), dtype=tf.resource, name=None)
wrote /home/nathan/Documents/a

/home/nathan/Documents/ariel-microflow-ml/building_tf/.venv/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1776950239.750705   29867 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1776950239.750720   29867 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1776950239.751000   29867 reader.cc:83] Reading SavedModel from: /tmp/tmp3287fu2m
I0000 00:00:1776950239.751319   29867 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1776950239.751324   29867 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmp3287fu2m
I0000 00:00:1776950239.754868   29867 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1776950239.755595   29867 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1776950239.783787   29867 loader.cc:220] Ru